In [ ]:
#Data set used to improve ad targeting strategies, optimize ad placement, and better understand user interaction with online advertisements

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/datasets/Train_amazon.csv')

In [ ]:
# Data preprocessing
    # data cleaning:
    # 1. Null values

df.isnull().sum()

,0
id,0
full_name,0
age,4288
gender,4212
device_type,1802
ad_position,1810
browsing_history,4295
time_of_day,1808
click,0


In [ ]:
# since null values are > 10% of data, we need to impute them
# 2. Handling null values

val = df['age'].mean()
val = val.round().astype(int)
val

np.int64(40)

In [ ]:
# a. imputing age with mean
df['age'].fillna(val, inplace = True)

In [ ]:
# b. imputing gender with mode
fill_val =df['gender'].mode()[0]
df['gender'].fillna(fill_val, inplace = True)

In [ ]:
# c. imputing device_type with mode
df['device_type'].fillna(df['device_type'].mode()[0], inplace = True)

In [ ]:
# d. imputing ad_position with mode
df['ad_position'].fillna(df['ad_position'].mode()[0], inplace=True)

In [ ]:
# e. Browsing history with mode
df['browsing_history'].fillna(df['browsing_history'].mode()[0], inplace=True)

In [ ]:
# f. time_of_day with mode
df['time_of_day'].fillna(df['time_of_day'].mode()[0], inplace = True)

In [ ]:
df.isnull().sum()

,0
id,0
full_name,0
age,0
gender,0
device_type,0
ad_position,0
browsing_history,0
time_of_day,0
click,0


In [ ]:
df.columns
# Converting categorical columns using Label encoder

Index(['id', 'full_name', 'age', 'gender', 'device_type', 'ad_position',
       'browsing_history', 'time_of_day', 'click'],
      dtype='object')

In [ ]:
from sklearn.preprocessing import LabelEncoder

cat = ['gender', 'device_type', 'ad_position', 'browsing_history', 'time_of_day']
d = {}

for c in cat:
    le = LabelEncoder()
    df[c] = le.fit_transform(df[c])
    original_classes = le.classes_

    # Create the mapping from encoded integer to original category name
    map_d = {original_classes[i]: i for i in range(len(original_classes))}

    print('classes', original_classes)
    print('mapping:', map_d)
    d.update(map_d)

classes ['Female' 'Male' 'Non-Binary']
mapping: {'Female': 0, 'Male': 1, 'Non-Binary': 2}
classes ['Desktop' 'Mobile' 'Tablet']
mapping: {'Desktop': 0, 'Mobile': 1, 'Tablet': 2}
classes ['Bottom' 'Side' 'Top']
mapping: {'Bottom': 0, 'Side': 1, 'Top': 2}
classes ['Education' 'Entertainment' 'News' 'Shopping' 'Social Media']
mapping: {'Education': 0, 'Entertainment': 1, 'News': 2, 'Shopping': 3, 'Social Media': 4}
classes ['Afternoon' 'Evening' 'Morning' 'Night']
mapping: {'Afternoon': 0, 'Evening': 1, 'Morning': 2, 'Night': 3}


In [ ]:
d

{'Female': 0,
 'Male': 1,
 'Non-Binary': 2,
 'Desktop': 0,
 'Mobile': 1,
 'Tablet': 2,
 'Bottom': 0,
 'Side': 1,
 'Top': 2,
 'Education': 0,
 'Entertainment': 1,
 'News': 2,
 'Shopping': 3,
 'Social Media': 4,
 'Afternoon': 0,
 'Evening': 1,
 'Morning': 2,
 'Night': 3}

In [ ]:
# droping id and user name
df.head(3)

,id,full_name,age,gender,device_type,ad_position,browsing_history,time_of_day,click
0,670,User670,22.0,0,0,2,3,0,1
1,3044,User3044,40.0,1,0,2,1,2,1
2,5912,User5912,41.0,2,0,1,0,3,1


In [ ]:
df.drop(['id', 'full_name'], axis = 1, inplace = True)

In [ ]:
df.head(3)

,age,gender,device_type,ad_position,browsing_history,time_of_day,click
0,22.0,0,0,2,3,0,1
1,40.0,1,0,2,1,2,1
2,41.0,2,0,1,0,3,1


In [ ]:
## 2. Standardisation
# split x(feature), y(label)

x = df.iloc[:, 0:-1]
y = df['click']

x.shape, y.shape


((9000, 6), (9000,))

In [ ]:
x.head(3)

,age,gender,device_type,ad_position,browsing_history,time_of_day
0,22.0,0,0,2,3,0
1,40.0,1,0,2,1,2
2,41.0,2,0,1,0,3


In [ ]:
from sklearn.preprocessing import StandardScaler
ss = StandardScaler()
x_df = ss.fit_transform(x)
x_df = pd.DataFrame(x_df)
x_df.head(3)

,0,1,2,3,4,5
0,-1.906665,-0.671967,-0.941143,1.460927,1.319881,-1.569676
1,-0.009230,0.640754,-0.941143,1.460927,-0.449001,0.404897
2,0.096184,1.953475,-0.941143,0.260055,-1.333443,1.392184


In [ ]:
# Model train test split

from sklearn.model_selection import train_test_split
xtrain,xtest,ytrain,ytest = train_test_split(x_df,y,test_size =0.2,random_state= 45)


In [ ]:
xtrain.shape, xtest.shape, ytrain.shape, ytest.shape

((7200, 6), (1800, 6), (7200,), (1800,))

In [ ]:
# model selection
# model 1: Logistic Regression
from sklearn.linear_model import LogisticRegression
lr_m1 = LogisticRegression(class_weight = 'balanced')
lr_m1.fit(xtrain,ytrain)

LogisticRegression(class_weight='balanced')

In [ ]:
# model1:  evaluation

pred1 = lr_m1.predict(xtest)
pred1


array([1, 0, 1, ..., 1, 0, 0])

In [ ]:
# model accuracy and classification report

from sklearn.metrics import accuracy_score, classification_report
accuracy_score(ytest,pred1)


0.5461111111111111

In [ ]:
print(classification_report(ytest,pred1))

              precision    recall  f1-score   support

           0       0.37      0.52      0.43       591
           1       0.70      0.56      0.62      1209

    accuracy                           0.55      1800
   macro avg       0.54      0.54      0.53      1800
weighted avg       0.59      0.55      0.56      1800



In [ ]:
## model 2: Decision tree classifier

from sklearn.tree import DecisionTreeClassifier
dt_m2 = DecisionTreeClassifier(class_weight= 'balanced', min_samples_split=2, random_state=42)
dt_m2.fit(xtrain,ytrain)

DecisionTreeClassifier(class_weight='balanced', random_state=42)

In [ ]:
# model2 evaluation
pred2 = dt_m2.predict(xtest)
pred2

array([0, 1, 0, ..., 1, 1, 1])

In [ ]:
accuracy_score(ytest,pred2)


0.7033333333333334

In [ ]:
print(classification_report(ytest,pred2))

              precision    recall  f1-score   support

           0       0.55      0.56      0.55       591
           1       0.78      0.77      0.78      1209

    accuracy                           0.70      1800
   macro avg       0.66      0.67      0.67      1800
weighted avg       0.70      0.70      0.70      1800



In [ ]:
# model 3: random forest classifier
from sklearn.ensemble import RandomForestClassifier
rf_m3 = RandomForestClassifier(class_weight= 'balanced', min_samples_split=2, random_state=60)
rf_m3.fit(xtrain,ytrain)

RandomForestClassifier(class_weight='balanced', random_state=60)

In [ ]:
# predicting model

pred3 = rf_m3.predict(xtest)
pred3

array([0, 1, 1, ..., 1, 1, 1])

In [ ]:
# evaluating
accuracy_score(ytest,pred3)

0.7083333333333334

In [ ]:
print(classification_report(ytest,pred3))

              precision    recall  f1-score   support

           0       0.56      0.52      0.54       591
           1       0.77      0.80      0.79      1209

    accuracy                           0.71      1800
   macro avg       0.67      0.66      0.66      1800
weighted avg       0.70      0.71      0.71      1800



In [ ]:
print("model 1:Logistic Regression", accuracy_score(ytest,pred1))
print("model 2:Decision Tree Classifier", accuracy_score(ytest,pred2)) # Decision Tree has slightly better precision for Click (1)
print("model 3:Random Forest Classifier", accuracy_score(ytest,pred3))

# the ML model should focus on precision more than recall since the goal is to improve ad targeting strategies, optimize ad placement,
# and better understand user interaction with online advertisements (this involves cost).
#if we were to focus on recall we will be wasting money if ads aren't clicked

# Hence Decision Tree model will be used

model 1:Logistic Regression 0.5461111111111111
model 2:Decision Tree Classifier 0.7033333333333334
model 3:Random Forest Classifier 0.7083333333333334


In [ ]:
## using test file to run in model 3

df1 = pd.read_csv('/content/drive/MyDrive/datasets/Test_amazon.csv')

In [ ]:
df1.head(3)

,id,full_name,age,gender,device_type,ad_position,browsing_history,time_of_day
0,5574,User5574,52.0,NaN,Desktop,Bottom,NaN,Afternoon
1,7652,User7652,NaN,Male,Mobile,Bottom,Education,NaN
2,3938,User3938,NaN,Male,Mobile,Bottom,NaN,Evening


In [ ]:
df1.shape

(1001, 8)

In [ ]:
final_df = df1[['id','full_name']]

In [ ]:
final_df

,id,full_name
0,5574,User5574
1,7652,User7652
2,3938,User3938
3,1424,User1424
4,4950,User4950
...,...,...
996,8510,User8510
997,7843,User7843
998,3914,User3914
999,7924,User7924


In [ ]:
test_df = df1.iloc[:,2: ]

In [ ]:
test_df

,age,gender,device_type,ad_position,browsing_history,time_of_day
0,52.0,NaN,Desktop,Bottom,NaN,Afternoon
1,NaN,Male,Mobile,Bottom,Education,NaN
2,NaN,Male,Mobile,Bottom,NaN,Evening
3,NaN,Male,NaN,Top,Entertainment,Afternoon
4,NaN,Male,Mobile,Side,NaN,NaN
...,...,...,...,...,...,...
996,NaN,NaN,Mobile,Top,Education,NaN
997,NaN,Female,Desktop,Bottom,Entertainment,NaN
998,NaN,Male,Mobile,Side,NaN,Morning
999,NaN,NaN,Desktop,NaN,Shopping,Morning


In [ ]:
# data preprocessing

test_df.isnull().sum()

,0
age,478
gender,482
device_type,198
ad_position,190
browsing_history,488
time_of_day,192


In [ ]:
# Imputing null values
# a.
val = test_df['age'].mean()

In [ ]:
test_df['age'].fillna(val,inplace = True)

In [ ]:
# b.
test_df['gender'].fillna(test_df['gender'].mode()[0], inplace = True)

In [ ]:
test_df['device_type'].fillna(test_df['device_type'].mode()[0], inplace = True)

In [ ]:
test_df['ad_position'].fillna(test_df['ad_position'].mode()[0], inplace = True)


In [ ]:
test_df['browsing_history'].fillna(test_df['browsing_history'].mode()[0], inplace = True)

In [ ]:
test_df['time_of_day'].fillna(test_df['time_of_day'].mode()[0], inplace = True)

In [ ]:
test_df.isnull().sum()

,0
age,0
gender,0
device_type,0
ad_position,0
browsing_history,0
time_of_day,0


In [ ]:
# labeling Categorical values

d = {'Female': 0,
 'Male': 1,
 'Non-Binary': 2,
 'Desktop': 0,
 'Mobile': 1,
 'Tablet': 2,
 'Bottom': 0,
 'Side': 1,
 'Top': 2,
 'Education': 0,
 'Entertainment': 1,
 'News': 2,
 'Shopping': 3,
 'Social Media': 4,
 'Afternoon': 0,
 'Evening': 1,
 'Morning': 2,
 'Night': 3}

In [ ]:
test_df.columns

Index(['age', 'gender', 'device_type', 'ad_position', 'browsing_history',
       'time_of_day'],
      dtype='object')

In [ ]:
cat = ['gender', 'device_type', 'ad_position', 'browsing_history','time_of_day']
for i in cat:
  test_df[i] = test_df[i].apply(lambda x: d[x])

In [ ]:
test_df

,age,gender,device_type,ad_position,browsing_history,time_of_day
0,52.000000,1,0,0,1,0
1,40.491396,1,1,0,0,2
2,40.491396,1,1,0,1,1
3,40.491396,1,1,2,1,0
4,40.491396,1,1,1,1,2
...,...,...,...,...,...,...
996,40.491396,1,1,2,0,2
997,40.491396,0,0,0,1,2
998,40.491396,1,1,1,1,2
999,40.491396,1,0,0,3,2


In [ ]:
# standard scaler is already called. from sklearn.preprocessing import StandardScaler
# ss = StandardScaler()

test_df = ss.fit_transform(test_df)
test_df = pd.DataFrame(test_df)
test_df

,0,1,2,3,4,5
0,1.201286e+00,-0.017094,-1.40098,-0.884844,-0.437941,-1.58571
1,2.296976e-17,-0.017094,0.00000,-0.884844,-1.300892,0.39717
2,2.296976e-17,-0.017094,0.00000,-0.884844,-0.437941,-0.59427
3,2.296976e-17,-0.017094,0.00000,1.545140,-0.437941,-1.58571
4,2.296976e-17,-0.017094,0.00000,0.330148,-0.437941,0.39717
...,...,...,...,...,...,...
996,2.296976e-17,-0.017094,0.00000,1.545140,-1.300892,0.39717
997,2.296976e-17,-1.728162,-1.40098,-0.884844,-0.437941,0.39717
998,2.296976e-17,-0.017094,0.00000,0.330148,-0.437941,0.39717
999,2.296976e-17,-0.017094,-1.40098,-0.884844,1.287961,0.39717


In [ ]:
# pred value using model 3(dt_m2)

f_pred = dt_m2.predict(test_df)
f_pred

array([0, 0, 1, ..., 0, 1, 0])

In [ ]:
final_df = pd.concat([final_df, pd.DataFrame(f_pred, columns = ['clicked'])], axis = 1)

In [ ]:
final_df.head(30)

,id,full_name,clicked
0,5574,User5574,0
1,7652,User7652,0
2,3938,User3938,1
3,1424,User1424,1
4,4950,User4950,0
5,5198,User5198,1
6,8891,User8891,1
7,1710,User1710,1
8,6744,User6744,0
9,3042,User3042,1


In [ ]:
final_df.to_csv('Amazon_submission.csv', index = False)